In [ ]:
# Cell 1: Environment, GIT_COMMIT, and Teacher Config Hash Assertions
import os
import sys
import subprocess
import hashlib
from pathlib import Path
import torch

# In Kaggle environment, install dependencies and set PYTHONPATH
if Path('/kaggle/input/wf052-code').exists():
    sys.path.insert(0, '/kaggle/input/wf052-code/src')
    os.environ['PYTHONPATH'] = '/kaggle/input/wf052-code/src:' + os.environ.get('PYTHONPATH', '')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytorchfire==1.0.3.post2', 'zarr', 'pyarrow'], check=True)
else:
    sys.path.insert(0, str(Path('src').resolve()))

from wf052.surrogate.model import WindConditionedUNet
from wf052.surrogate.standin_generator import EXPECTED_TEACHER_CONFIG_HASH, EXPECTED_CANONICAL_SIM_HASH

print('=' * 75)
print('CELL 1: ENVIRONMENT & GIT COMMIT VERIFICATION')
print('=' * 75)

# Check compute device
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else 'cpu')
print(f'Active Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Device Name: {torch.cuda.get_device_name(0)}')
    print(f'CUDA Version: {torch.version.cuda}')

# Verify GIT_COMMIT.txt
git_commit_file = None
for p in [Path('GIT_COMMIT.txt'), Path('/kaggle/input/wf052-code/GIT_COMMIT.txt')]:
    if p.exists():
        git_commit_file = p
        break

if git_commit_file:
    git_info = git_commit_file.read_text(encoding='utf-8').strip()
    print(f'GIT_COMMIT.txt found at {git_commit_file}:\n{git_info}')
else:
    try:
        commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
        status = subprocess.check_output(['git', 'status', '--porcelain'], text=True).strip()
        print(f'Git repository HEAD:\ncommit: {commit}\ndirty: {bool(status)}')
    except Exception as exc:
        print(f'Git commit lookup: unknown ({exc})')

# Verify Model Parameter Count
model = WindConditionedUNet()
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable Parameters: {n_params:,} (Must be exactly 4,094,852)')
assert n_params == 4094852, f'Parameter count assertion failed: {n_params} != 4094852'

# Verify Teacher Config Hash
teacher_cfg = None
for p in [Path('configs/teacher_v2.yaml'), Path('/kaggle/input/wf052-code/configs/teacher_v2.yaml'), Path('out/wf052-teacher-config/teacher_v2_a6a92f81.yaml')]:
    if p.exists():
        teacher_cfg = p
        break

if teacher_cfg:
    content = teacher_cfg.read_bytes()
    cfg_hash = hashlib.sha256(content).hexdigest()
    print(f'Teacher Config ({teacher_cfg}): SHA-256 = {cfg_hash}')
    assert cfg_hash == EXPECTED_TEACHER_CONFIG_HASH, f'Teacher config hash mismatch: {cfg_hash} != {EXPECTED_TEACHER_CONFIG_HASH}'
else:
    print(f'Expected Teacher Config Hash: {EXPECTED_TEACHER_CONFIG_HASH}')


In [ ]:
# Cell 2: Data Check (Counts per Fire & Bin, Split Assertions)
import os
import yaml
from pathlib import Path
import pandas as pd
import numpy as np

from wf052.surrogate.dataset import CompactSurrogateDataset
from wf052.surrogate.standin_generator import validate_shard_metadata

print('=' * 75)
print('CELL 2: DATA INTEGRITY & SPLIT VERIFICATION')
print('=' * 75)

# Locate shards directory
if Path('/kaggle/input/wf052-shards').exists():
    shards_dir = Path('/kaggle/input/wf052-shards')
elif Path('data/shards_smoke_2i').exists():
    shards_dir = Path('data/shards_smoke_2i')
else:
    shards_dir = Path(os.environ.get('SHARDS_DIR', 'data/shards_smoke_2i'))

# Locate tiles directory
if Path('/kaggle/input/wf052-tiles-train/tiles/fires').exists():
    tiles_dir = Path('/kaggle/input/wf052-tiles-train/tiles/fires')
elif Path('data/tiles/fires').exists():
    tiles_dir = Path('data/tiles/fires')
else:
    tiles_dir = Path(os.environ.get('TILES_DIR', 'data/tiles/fires'))

print(f'Shards Directory: {shards_dir.resolve()}')
print(f'Tiles Directory:  {tiles_dir.resolve()}')

# Load splits from scenarios.yaml
scenarios_cfg_file = None
for p in [Path('configs/scenarios.yaml'), Path('/kaggle/input/wf052-code/configs/scenarios.yaml')]:
    if p.exists():
        scenarios_cfg_file = p
        break

assert scenarios_cfg_file is not None, 'configs/scenarios.yaml not found!'
with open(scenarios_cfg_file, 'r', encoding='utf-8') as f:
    scenarios_cfg = yaml.safe_load(f)

splits = scenarios_cfg.get('splits', {})
train_fires = splits.get('train', ['kincade', 'glass', 'zogg', 'creek'])
val_fires = splits.get('val', ['bobcat'])
test_fires = splits.get('test', ['caldor', 'tamarack'])

print(f'Configured Splits: Train={train_fires}, Val={val_fires}, Test={test_fires}')

# STRICT DATA LEAKAGE ASSERTIONS: Fail immediately if any fire appears in two splits
set_train = set(train_fires)
set_val = set(val_fires)
set_test = set(test_fires)

assert set_train.isdisjoint(set_val), f'DATA LEAKAGE: Fire shared between train and val: {set_train & set_val}'
assert set_train.isdisjoint(set_test), f'DATA LEAKAGE: Fire shared between train and test: {set_train & set_test}'
assert set_val.isdisjoint(set_test), f'DATA LEAKAGE: Fire shared between val and test: {set_val & set_test}'
print('Split Disjointness Assertion: PASSED (Zero data leakage across splits)')

# Load Compact Dataset and Validate Metadata
ds = CompactSurrogateDataset(shard_paths=shards_dir, fires_base_dir=tiles_dir)
print(f'Loaded Shards: {len(ds.shard_paths)} shards, {len(ds)} total samples')

for sp in ds.shard_paths:
    validate_shard_metadata(sp)
print('Shard Metadata Validation: PASSED (Hashes and schema_version verified)')

# Per-fire and Per-bin Counts
records = []
for idx, f in enumerate(ds.fire_ids):
    bp = ds[idx]['burn_prob'].numpy()
    mean_bf = float(np.mean(bp))
    bin_k = int(np.digitize(mean_bf, [0.002, 0.02, 0.1, 0.5]))
    records.append({'fire': f, 'bin_kmean': bin_k})

df_counts = pd.DataFrame(records)
print('\n--- Shard Sample Counts (Fire x Bin_kmean) ---')
ct = pd.crosstab(df_counts['fire'], df_counts['bin_kmean'], margins=True)
print(ct)

train_idx = [i for i, f in enumerate(ds.fire_ids) if f in train_fires]
val_idx = [i for i, f in enumerate(ds.fire_ids) if f in val_fires]
test_idx = [i for i, f in enumerate(ds.fire_ids) if f in test_fires]
print(f'\nSamples per Split: Train={len(train_idx)}, Val={len(val_idx)}, Test={len(test_idx)}')


In [ ]:
# Cell 3: Surrogate Model Training with Multi-Task Loss & Checkpointing
import time
import random
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Subset

from wf052.surrogate.dataset import collate_surrogate_batch
from wf052.surrogate.losses import SurrogateMultiTaskLoss
from wf052.surrogate.evaluate import evaluate_model_on_dataset, aggregate_per_bin_metrics

print('=' * 75)
print('CELL 3: SURROGATE TRAINING LOOP')
print('=' * 75)

epochs = int(os.environ.get('EPOCHS', 5))
batch_size = int(os.environ.get('BATCH', 8))
lr = float(os.environ.get('LR', 3e-4))
seed = int(os.environ.get('SEED', 42000))
max_minutes = float(os.environ.get('MAX_MINUTES', 240.0))
amp = (int(os.environ.get('AMP', 0)) == 1) and torch.cuda.is_available()

# Reproducibility
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
if hasattr(torch, 'mps') and torch.backends.mps.is_available():
    torch.mps.manual_seed(seed)

ckpt_dir = Path('checkpoints')
ckpt_dir.mkdir(parents=True, exist_ok=True)
rep_dir = Path('reports')
rep_dir.mkdir(parents=True, exist_ok=True)
last_pt = ckpt_dir / 'last.pt'
best_pt = ckpt_dir / 'best.pt'

model = WindConditionedUNet().to(device)
optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=max(1, epochs), eta_min=1e-6)
loss_fn = SurrogateMultiTaskLoss(quantiles=(0.1, 0.5, 0.9), lambda_order=1.0, lambda_bce=0.5, horizon_scale=720.0)

train_loader = DataLoader(Subset(ds, train_idx), batch_size=batch_size, shuffle=True, collate_fn=collate_surrogate_batch)
val_loader = DataLoader(Subset(ds, val_idx), batch_size=batch_size, shuffle=False, collate_fn=collate_surrogate_batch)

best_val_loss = float('inf')
patience = 6
patience_counter = 0
train_history = []
step_times = []
t_start = time.perf_counter()

print(f'Training Configuration: Epochs={epochs}, Batch={batch_size}, LR={lr}, AMP={amp}, Device={device}')

for ep in range(epochs):
    if (time.perf_counter() - t_start) / 60.0 >= max_minutes:
        print(f'Reached MAX_MINUTES={max_minutes} cap. Stopping.')
        break

    t_ep = time.perf_counter()
    model.train()
    tr_losses = {'total': 0.0, 'q10': 0.0, 'q50': 0.0, 'q90': 0.0, 'order': 0.0, 'bce': 0.0}
    n_batches = 0

    for batch in train_loader:
        t_s = time.perf_counter()
        spatial = batch.spatial_features.to(device)
        wind = batch.wind_series.to(device)
        t_q = batch.target_quantiles.to(device)
        c_q = batch.censor_quantiles.to(device)
        bp = batch.burn_prob.to(device)

        optimizer.zero_grad()
        if amp:
            with torch.cuda.amp.autocast():
                out = model(spatial, wind)
                lb = loss_fn(out.quantiles, out.burn_logits, target_quantiles=t_q, censor_quantiles=c_q, burn_prob=bp)
        else:
            out = model(spatial, wind)
            lb = loss_fn(out.quantiles, out.burn_logits, target_quantiles=t_q, censor_quantiles=c_q, burn_prob=bp)

        lb.total_loss.backward()
        optimizer.step()

        if len(step_times) < 50:
            step_times.append(time.perf_counter() - t_s)

        tr_losses['total'] += float(lb.total_loss.item())
        tr_losses['q10'] += float(lb.loss_q10.item())
        tr_losses['q50'] += float(lb.loss_q50.item())
        tr_losses['q90'] += float(lb.loss_q90.item())
        tr_losses['order'] += float(lb.ordering_loss.item())
        tr_losses['bce'] += float(lb.bce_loss.item())
        n_batches += 1

    scheduler.step()
    mean_tr = {k: v / max(1, n_batches) for k, v in tr_losses.items()}

    # Validation loss
    model.eval()
    val_tot = 0.0
    val_batches = 0
    with torch.no_grad():
        for batch in val_loader:
            spatial = batch.spatial_features.to(device)
            wind = batch.wind_series.to(device)
            t_q = batch.target_quantiles.to(device)
            c_q = batch.censor_quantiles.to(device)
            bp = batch.burn_prob.to(device)
            out = model(spatial, wind)
            lb = loss_fn(out.quantiles, out.burn_logits, target_quantiles=t_q, censor_quantiles=c_q, burn_prob=bp)
            val_tot += float(lb.total_loss.item())
            val_batches += 1

    mean_val_tot = val_tot / max(1, val_batches)
    df_val_bins = aggregate_per_bin_metrics(evaluate_model_on_dataset(model, ds, target_indices=val_idx, device=device))
    ep_sec = time.perf_counter() - t_ep

    print(f'Epoch {ep:2d}/{epochs} ({ep_sec:.1f}s) | Train Total: {mean_tr["total"]:.5f} | Val Total: {mean_val_tot:.5f} | Val Brier: {df_val_bins["brier"].mean():.4f}')

    # Save last.pt atomically
    ckpt_data = {
        'epoch': ep,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_loss': best_val_loss,
    }
    tmp_last = ckpt_dir / 'last.pt.tmp'
    torch.save(ckpt_data, tmp_last)
    os.replace(tmp_last, last_pt)

    # Save best.pt
    if mean_val_tot < best_val_loss:
        best_val_loss = mean_val_tot
        patience_counter = 0
        tmp_best = ckpt_dir / 'best.pt.tmp'
        torch.save(ckpt_data, tmp_best)
        os.replace(tmp_best, best_pt)
        print(f'  --> Saved new best model to {best_pt} (val_loss={best_val_loss:.5f})')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f'Early stopping triggered after {patience} epochs without improvement.')
            break

    train_history.append({'epoch': ep, 'train_total_loss': mean_tr['total'], 'val_total_loss': mean_val_tot, 'val_brier': df_val_bins['brier'].mean()})

pd.DataFrame(train_history).to_csv('reports/train_log.csv', index=False)
if step_times:
    mean_ms = np.mean(step_times) * 1000.0
    p95_ms = np.percentile(step_times, 95) * 1000.0
    print(f'\nStep Latency: Mean={mean_ms:.1f} ms, P95={p95_ms:.1f} ms')


In [ ]:
# Cell 4: Validation & Held-out Test Evaluation (Surrogate vs Baselines B0 & B1)
import torch
import pandas as pd

from wf052.surrogate.evaluate import (
    evaluate_model_on_dataset,
    evaluate_baseline_on_dataset,
    aggregate_per_bin_metrics,
    fit_baseline_b0,
    fit_baseline_b1,
)

print('=' * 75)
print('CELL 4: EVALUATION ON VALIDATION & TEST SPLITS')
print('=' * 75)

# Load best checkpoint
best_pt = Path('checkpoints/best.pt')
if best_pt.exists():
    ckpt = torch.load(best_pt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded best model checkpoint from Epoch {ckpt["epoch"]}')

# 1. Validation Evaluation
print('\n--- 1. Validation Set Evaluation (Fire: bobcat) ---')
df_val = evaluate_model_on_dataset(model, ds, target_indices=val_idx, device=device)
val_table = aggregate_per_bin_metrics(df_val)
print(val_table.to_string(index=False))

# 2. Held-out Test Evaluation
print('\n--- 2. Held-out Test Evaluation (Fires: caldor, tamarack) ---')
df_test = evaluate_model_on_dataset(model, ds, target_indices=test_idx, device=device)
test_surrogate = aggregate_per_bin_metrics(df_test)
test_surrogate['model'] = 'Surrogate (UNet)'

# Baselines on test set
b0 = fit_baseline_b0(ds, train_fires=train_fires)
b1 = fit_baseline_b1(ds, train_fires=train_fires)

df_b0 = evaluate_baseline_on_dataset(b0, ds, test_idx, baseline_name='B0')
test_b0 = aggregate_per_bin_metrics(df_b0)
test_b0['model'] = 'Baseline B0'

df_b1 = evaluate_baseline_on_dataset(b1, ds, test_idx, baseline_name='B1')
test_b1 = aggregate_per_bin_metrics(df_b1)
test_b1['model'] = 'Baseline B1'

df_combined = pd.concat([test_surrogate, test_b0, test_b1], ignore_index=True)
cols_show = ['model', 'bin', 'bin_name', 'n', 'flag', 'brier', 'iou', 'mae_q50', 'false_unburned_share']
print(df_combined[cols_show].to_string(index=False))

df_combined.to_csv('reports/test_eval.csv', index=False)
print('\nSaved final test evaluation to reports/test_eval.csv')


In [ ]:
# Cell 5: Output Listing and SHA-256 Checksums
import hashlib
from pathlib import Path

print('=' * 75)
print('CELL 5: OUTPUT ARTIFACT LISTING & SHA-256 CHECKSUMS')
print('=' * 75)

out_files = [
    Path('checkpoints/best.pt'),
    Path('checkpoints/last.pt'),
    Path('reports/train_log.csv'),
    Path('reports/test_eval.csv'),
]

for p in out_files:
    if p.exists():
        data = p.read_bytes()
        sha256 = hashlib.sha256(data).hexdigest()
        size_kb = len(data) / 1024.0
        print(f'{str(p):25s} | {size_kb:8.2f} KB | SHA-256: {sha256}')
    else:
        print(f'{str(p):25s} | NOT FOUND')
